# Srf d/d (Pgr cre/+ Srf f/f) DEG Analysis

Identify and analyze the differentially expressed gene (DEG) lists present between Srf flox and Srf knockout (Srf d/d) samples.

## Step 0: Set up environment and load R

**At this step, we need to change to the R kernel if we are using Jupyter for this.** (You can also just use R directly.) Make sure you install R and the packages below. We will use them at various points in the analysis.

In [1]:
library(DESeq2)
library(data.table)
library(tidyr)
library(dplyr)

Loading required package: S4Vectors

Loading required package: stats4

Loading required package: BiocGenerics


Attaching package: ‘BiocGenerics’


The following objects are masked from ‘package:stats’:

    IQR, mad, sd, var, xtabs


The following objects are masked from ‘package:base’:

    anyDuplicated, aperm, append, as.data.frame, basename, cbind,
    colnames, dirname, do.call, duplicated, eval, evalq, Filter, Find,
    get, grep, grepl, intersect, is.unsorted, lapply, Map, mapply,
    match, mget, order, paste, pmax, pmax.int, pmin, pmin.int,
    Position, rank, rbind, Reduce, rownames, sapply, setdiff, sort,
    table, tapply, union, unique, unsplit, which.max, which.min



Attaching package: ‘S4Vectors’


The following object is masked from ‘package:utils’:

    findMatches


The following objects are masked from ‘package:base’:

    expand.grid, I, unname


Loading required package: IRanges

Loading required package: GenomicRanges

Loading required package: GenomeInfoDb

Loa

In [5]:
sessionInfo()

R version 4.3.1 (2023-06-16)
Platform: x86_64-conda-linux-gnu (64-bit)
Running under: Rocky Linux 8.10 (Green Obsidian)

Matrix products: default
BLAS/LAPACK: /ddn/gs1/home/marquardtrm/miniforge3/envs/r431/lib/libopenblasp-r0.3.30.so;  LAPACK version 3.12.0

locale:
 [1] LC_CTYPE=en_US.UTF-8       LC_NUMERIC=C              
 [3] LC_TIME=en_US.UTF-8        LC_COLLATE=en_US.UTF-8    
 [5] LC_MONETARY=en_US.UTF-8    LC_MESSAGES=en_US.UTF-8   
 [7] LC_PAPER=en_US.UTF-8       LC_NAME=C                 
 [9] LC_ADDRESS=C               LC_TELEPHONE=C            
[11] LC_MEASUREMENT=en_US.UTF-8 LC_IDENTIFICATION=C       

time zone: America/New_York
tzcode source: system (glibc)

attached base packages:
[1] stats4    stats     graphics  grDevices utils     datasets  methods  
[8] base     

other attached packages:
 [1] dplyr_1.1.4                 tidyr_1.3.1                
 [3] data.table_1.17.8           DESeq2_1.42.1              
 [5] SummarizedExperiment_1.32.0 Biobase_2.62.0            

## Step 1: Read in and process counts data

Load the count data into regular matrices that are amenable to principal component analysis and differential gene expression matrices.

In [7]:
# Load in the table and make sure it works/reads fine!
# There's a comment at the beginning of the counts file, so skip that first line.
filePath = "Counts-SRF-NonNormalized-NCBIRefSeq.txt"
cts = fread(file=filePath)
head(cts)

Geneid,SrfFlox1,SrfFlox2,SrfFlox3,SrfFlox4,SrfKO1,SrfKO2,SrfKO3,SrfKO4
<chr>,<int>,<int>,<int>,<int>,<int>,<int>,<int>,<int>
TrnP,531,297,490,248,1046,667,550,460
CYTB,285672,126045,239264,87972,406537,323316,240628,204193
ND6,44256,22591,41514,16800,77474,52117,41620,36438
ND5,122325,63836,121834,43150,216536,148128,112433,97113
TrnL2,10,9,22,0,28,2,1,2
TrnH,22,30,39,4,38,4,3,5


In [8]:
# Make the DESeq counts matrix.
dcts = as.data.frame(cts[,2:ncol(cts)])
rownames(dcts) = unlist(cts[,'Geneid'])
head(dcts)

,SrfFlox1,SrfFlox2,SrfFlox3,SrfFlox4,SrfKO1,SrfKO2,SrfKO3,SrfKO4
,<int>,<int>,<int>,<int>,<int>,<int>,<int>,<int>
TrnP,531,297,490,248,1046,667,550,460
CYTB,285672,126045,239264,87972,406537,323316,240628,204193
ND6,44256,22591,41514,16800,77474,52117,41620,36438
ND5,122325,63836,121834,43150,216536,148128,112433,97113
TrnL2,10,9,22,0,28,2,1,2
TrnH,22,30,39,4,38,4,3,5


To also perform the differential expression analysis, DESeq needs information about certain discrete variables (factors) in each sample. I've made a metadata table below.

In [13]:
# Make a factor list between control and experimental groups.
# We can do this by just stripping the numbers out of the group names.
column_names <- colnames(dcts)
groups = as.factor(sub("\\d", "", column_names))
groups = relevel(groups, ref="SrfFlox")
dgroups = data.frame(row.names=column_names, Group=groups)
dgroups
levels(groups)

,Group
,<fct>
SrfFlox1,SrfFlox
SrfFlox2,SrfFlox
SrfFlox3,SrfFlox
SrfFlox4,SrfFlox
SrfKO1,SrfKO
SrfKO2,SrfKO
SrfKO3,SrfKO
SrfKO4,SrfKO


[1] "SrfFlox" "SrfKO"

A concrete check to ensure the sample names in both data and metadata files are the same and ordered the same.

In [14]:
# Ensure the data and metadata labels are within each other and in the same order.
all(rownames(groups) %in% colnames(dcts))
all(rownames(groups) == colnames(dcts))

[1] TRUE

[1] TRUE

## Step 2: Run DESeq2

Make the DESeq dataset...

In [15]:
# Make the DESeqDataSet and analyze.
dds = DESeqDataSetFromMatrix(countData = dcts,
                             colData = dgroups,
                             design = ~ Group)
dds

class: DESeqDataSet 
dim: 22952 8 
metadata(1): version
assays(1): counts
rownames(22952): TrnP CYTB ... Gm7341 Xkr4
rowData names(0):
colnames(8): SrfFlox1 SrfFlox2 ... SrfKO3 SrfKO4
colData names(1): Group

... and perform differential expression analysis.

In [16]:
dds = DESeq(dds)
res = results(dds)
res

estimating size factors

estimating dispersions

gene-wise dispersion estimates

mean-dispersion relationship

final dispersion estimates

fitting model and testing



log2 fold change (MLE): Group SrfKO vs SrfFlox 
Wald test p-value: Group SrfKO vs SrfFlox 
DataFrame with 22952 rows and 6 columns
          baseMean log2FoldChange     lfcSE      stat      pvalue      padj
         <numeric>      <numeric> <numeric> <numeric>   <numeric> <numeric>
TrnP   4.92647e+02      0.0824846 0.1351007  0.610542    0.541503  0.815380
CYTB   2.16128e+05      0.0513075 0.0477271  1.075018    0.282367  0.620318
ND6    3.78441e+04      0.0601788 0.0759440  0.792409    0.428122  0.744967
ND5    1.04423e+05      0.0486902 0.0733605  0.663711    0.506876  0.794299
TrnL2  7.67087e+00     -1.1409576 1.0666918 -1.069623    0.284789        NA
...            ...            ...       ...       ...         ...       ...
Gm7357  1474.82001      0.2676692  0.105704  2.532259 0.011333013 0.0990939
Sox17  13677.68782      0.0456530  0.107411  0.425033 0.670812667 0.8840302
Rp1     2181.32621     -0.0129024  0.109069 -0.118296 0.905833084 0.9704026
Gm7341     6.59115      0.1524537

## Step 3: Generate DEG list

To facilitate the export and analysis of these data in other programs, generate a list of highly differentially expressed genes.

We also do the following to include a filtered gene expression list: list everything with p-adjusted < 0.05, sort by least p-value, and package it into a CSV.

In [21]:
# Carry out the above instructions.
testCounts = counts(dds)
testCounts = cbind(testCounts, res[['padj']], 2 ** res[['log2FoldChange']])
colnames(testCounts) = c(colnames(testCounts)[1:8], 'padj', 'FC')
testCountsdf = as.data.frame(testCounts)
testCountsdf = testCountsdf[!is.na(testCountsdf[['FC']]),]
head(testCountsdf)

,SrfFlox1,SrfFlox2,SrfFlox3,SrfFlox4,SrfKO1,SrfKO2,SrfKO3,SrfKO4,padj,FC
,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>
TrnP,531,297,490,248,1046,667,550,460,0.8153795,1.0588400
CYTB,285672,126045,239264,87972,406537,323316,240628,204193,0.6203180,1.0362036
ND6,44256,22591,41514,16800,77474,52117,41620,36438,0.7449669,1.0425949
ND5,122325,63836,121834,43150,216536,148128,112433,97113,0.7942987,1.0343254
TrnL2,10,9,22,0,28,2,1,2,NA,0.4534585
TrnH,22,30,39,4,38,4,3,5,0.1125794,0.2822867


In [22]:
temp = testCountsdf[['FC']]
temp[temp < 1] = -1 / temp[temp < 1]
testCountsdf[['FC']] = temp
head(testCountsdf)

,SrfFlox1,SrfFlox2,SrfFlox3,SrfFlox4,SrfKO1,SrfKO2,SrfKO3,SrfKO4,padj,FC
,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>
TrnP,531,297,490,248,1046,667,550,460,0.8153795,1.058840
CYTB,285672,126045,239264,87972,406537,323316,240628,204193,0.6203180,1.036204
ND6,44256,22591,41514,16800,77474,52117,41620,36438,0.7449669,1.042595
ND5,122325,63836,121834,43150,216536,148128,112433,97113,0.7942987,1.034325
TrnL2,10,9,22,0,28,2,1,2,NA,-2.205274
TrnH,22,30,39,4,38,4,3,5,0.1125794,-3.542498


In [23]:
testCountsThresholdeddf = subset(testCountsdf, padj < 0.05)
testCountsThresholdeddf = testCountsThresholdeddf[order(testCountsThresholdeddf[['FC']], decreasing=TRUE),]

head(testCountsThresholdeddf)
tail(testCountsThresholdeddf)
nrow(testCountsThresholdeddf)

,SrfFlox1,SrfFlox2,SrfFlox3,SrfFlox4,SrfKO1,SrfKO2,SrfKO3,SrfKO4,padj,FC
,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>
Gm32592,6,1,8,0,83,66,28,28,2.557404e-06,8.979192
Gm7347,34,11,19,6,201,150,158,79,7.366332e-13,5.665944
Aqp8,32,10,9,9,119,150,107,93,5.268956e-09,5.269960
Apoc3,4,2,8,6,90,8,50,27,2.307553e-02,5.002339
Vwa5b1,7,4,12,4,120,8,50,21,3.321669e-02,4.356306
Sstr2,10,2,11,2,43,42,44,24,1.046804e-03,4.300878


,SrfFlox1,SrfFlox2,SrfFlox3,SrfFlox4,SrfKO1,SrfKO2,SrfKO3,SrfKO4,padj,FC
,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>
Gm52666,25,36,62,5,23,12,5,2,4.998323e-03,-5.272366
Thsd7b,889,365,795,98,227,167,70,109,8.143053e-14,-5.384855
Rps8-ps2,84,88,69,4,56,9,8,4,7.854636e-03,-5.873277
Slc5a5,5187,2060,4893,119,1263,774,184,650,2.185869e-02,-5.878661
Acod1,4003,1541,3694,110,898,557,120,318,5.425737e-03,-7.074020
Gm43700,423,558,759,10,102,13,25,59,5.318259e-03,-13.229763


[1] 1670

In [24]:
# Output results to a CSV
fwrite(testCountsThresholdeddf, "./Srf_DEGs_P05.csv", row.names=TRUE)

In [25]:
# Make DEseq2 normalized counts table
normalized_counts_table <- counts(dds, normalized = TRUE)
head(normalized_counts_table)

,SrfFlox1,SrfFlox2,SrfFlox3,SrfFlox4,SrfKO1,SrfKO2,SrfKO3,SrfKO4
TrnP,3.957382e+02,501.86726,439.77817,5.877726e+02,542.35958,4.500274e+02,5.365275e+02,4.871039e+02
CYTB,2.129027e+05,212989.42485,214740.98928,2.084981e+05,210792.77011,2.181425e+05,2.347337e+05,2.162244e+05
ND6,3.298265e+04,38174.01798,37259.08381,3.981685e+04,40170.90467,3.516353e+04,4.060050e+04,3.858498e+04
ND5,9.116511e+04,107869.35558,109346.80390,1.022677e+05,112275.69267,9.994251e+04,1.096789e+05,1.028350e+05
TrnL2,7.452697e+00,15.20810,19.74514,0.000000e+00,14.51823,1.349407e+00,9.755046e-01,2.117843e+00
TrnH,1.639593e+01,50.69366,35.00275,9.480203e+00,19.70331,2.698815e+00,2.926514e+00,5.294608e+00


In [27]:
# Output results to a CSV
write.csv(normalized_counts_table, "./Srf_normcounts.csv", row.names=TRUE)

In [28]:
# to make sure we have all the DEseq2 output, let's convert all the results to a dataframe

res_df <- as.data.frame(res)
head(res_df)

,baseMean,log2FoldChange,lfcSE,stat,pvalue,padj
,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>
TrnP,4.926468e+02,0.08248457,0.13510067,0.6105415,0.54150313,0.8153795
CYTB,2.161281e+05,0.05130749,0.04772708,1.0750183,0.28236655,0.6203180
ND6,3.784407e+04,0.06017876,0.07594405,0.7924092,0.42812212,0.7449669
ND5,1.044226e+05,0.04869016,0.07336054,0.6637105,0.50687561,0.7942987
TrnL2,7.670865e+00,-1.14095761,1.06669179,-1.0696226,0.28478924,NA
TrnH,1.777447e+01,-1.82476714,0.74193468,-2.4594714,0.01391418,0.1125794


In [29]:
# Output results to a CSV
write.csv(res_df, "./Srf_DEseq2Results.csv", row.names=TRUE)

# MyoPgr d/d (Myh11 cre/+ Pgr f/f) DEG Analysis

Identify and analyze the differentially expressed gene (DEG) lists present between Pgr flox and Pgr muscle knockout (MyoPgr d/d) samples.

## Step 1: Read in and process counts data

Load the count data into regular matrices that are amenable to principal component analysis and differential gene expression matrices.

In [30]:
# Now let's repeat everything for the MyoPgr knockout data

# Load in the table and make sure it works/reads fine!
# There's a comment at the beginning of the counts file, so skip that first line.
filePath = "Counts-PGR-NonNormalized-NCBIRefSeq.txt"
cts = fread(file=filePath)
head(cts)

Geneid,PgrFlox1,PgrFlox2,PgrFlox3,PgrKO1,PgrKO2,PgrKO3
<chr>,<int>,<int>,<int>,<int>,<int>,<int>
TrnP,7827,9648,7676,6926,6661,7954
TrnT,46,57,48,30,42,50
CYTB,2818357,3056764,3627036,2714250,3060874,3061130
TrnE,181,272,331,190,241,307
ND6,360990,486772,501792,381434,419012,419213
ND5,1264726,1503209,1659052,1104796,1434887,1482613


In [31]:
# Make the DESeq counts matrix.
dcts = as.data.frame(cts[,2:ncol(cts)])
rownames(dcts) = unlist(cts[,'Geneid'])
head(dcts)

,PgrFlox1,PgrFlox2,PgrFlox3,PgrKO1,PgrKO2,PgrKO3
,<int>,<int>,<int>,<int>,<int>,<int>
TrnP,7827,9648,7676,6926,6661,7954
TrnT,46,57,48,30,42,50
CYTB,2818357,3056764,3627036,2714250,3060874,3061130
TrnE,181,272,331,190,241,307
ND6,360990,486772,501792,381434,419012,419213
ND5,1264726,1503209,1659052,1104796,1434887,1482613


In [32]:
# Make a factor list between control and experimental groups.
# We can do this by just stripping the numbers out of the group names.
column_names <- colnames(dcts)
groups = as.factor(sub("\\d", "", column_names))
groups = relevel(groups, ref="PgrFlox")
dgroups = data.frame(row.names=column_names, Group=groups)
dgroups
levels(groups)

,Group
,<fct>
PgrFlox1,PgrFlox
PgrFlox2,PgrFlox
PgrFlox3,PgrFlox
PgrKO1,PgrKO
PgrKO2,PgrKO
PgrKO3,PgrKO


[1] "PgrFlox" "PgrKO"

In [33]:
# Ensure the data and metadata labels are within each other and in the same order.
all(rownames(groups) %in% colnames(dcts))
all(rownames(groups) == colnames(dcts))

[1] TRUE

[1] TRUE

## Step 2: Run DESeq2

Make the DESeq dataset...

In [34]:
# Make the DESeqDataSet and analyze.
dds = DESeqDataSetFromMatrix(countData = dcts,
                             colData = dgroups,
                             design = ~ Group)
dds

class: DESeqDataSet 
dim: 30875 6 
metadata(1): version
assays(1): counts
rownames(30875): TrnP TrnT ... Gm7341 Xkr4
rowData names(0):
colnames(6): PgrFlox1 PgrFlox2 ... PgrKO2 PgrKO3
colData names(1): Group

In [35]:
dds = DESeq(dds)
res = results(dds)
res

estimating size factors

estimating dispersions

gene-wise dispersion estimates

mean-dispersion relationship

final dispersion estimates

fitting model and testing



log2 fold change (MLE): Group PgrKO vs PgrFlox 
Wald test p-value: Group PgrKO vs PgrFlox 
DataFrame with 30875 rows and 6 columns
          baseMean log2FoldChange     lfcSE       stat     pvalue      padj
         <numeric>      <numeric> <numeric>  <numeric>  <numeric> <numeric>
TrnP   7.78555e+03    -0.15976155 0.1934733 -0.8257552   0.408943  0.709940
TrnT   4.52003e+01    -0.24867198 0.3926308 -0.6333482   0.526506  0.791325
CYTB   3.03453e+06    -0.02070209 0.0725288 -0.2854329   0.775313  0.919828
TrnE   2.49170e+02     0.00521698 0.2473106  0.0210949   0.983170  0.994703
ND6    4.24912e+05    -0.05939289 0.1236720 -0.4802453   0.631053  0.850714
...            ...            ...       ...        ...        ...       ...
Sox17  125344.0783      -0.487217  0.275313  -1.769682 0.07678012 0.3082556
Gm6101     24.3263      -0.767032  0.827784  -0.926609 0.35412955        NA
Rp1     23353.7536      -0.154830  0.135913  -1.139181 0.25462747 0.5732241
Gm7341     65.5945      -0.588756

## Step 3: Generate DEG list

To facilitate the export and analysis of these data in other programs, generate a list of highly differentially expressed genes.

We also do the following to include a filtered gene expression list: list everything with p-adjusted < 0.05, sort by least p-value, and package it into a CSV.

In [37]:
# include a filtered gene expression list: list everything with p-adjusted < 0.05, sort by least p-value, and package it into a CSV.

testCounts = counts(dds)
testCounts = cbind(testCounts, res[['padj']], 2 ** res[['log2FoldChange']])
colnames(testCounts) = c(colnames(testCounts)[1:6], 'padj', 'FC')
testCountsdf = as.data.frame(testCounts)
testCountsdf = testCountsdf[!is.na(testCountsdf[['FC']]),]
head(testCountsdf)

,PgrFlox1,PgrFlox2,PgrFlox3,PgrKO1,PgrKO2,PgrKO3,padj,FC
,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>
TrnP,7827,9648,7676,6926,6661,7954,0.7099399,0.8951730
TrnT,46,57,48,30,42,50,0.7913255,0.8416708
CYTB,2818357,3056764,3627036,2714250,3060874,3061130,0.9198277,0.9857529
TrnE,181,272,331,190,241,307,0.9947032,1.0036227
ND6,360990,486772,501792,381434,419012,419213,0.8507135,0.9596679
ND5,1264726,1503209,1659052,1104796,1434887,1482613,0.8273932,0.9584622


In [38]:
temp = testCountsdf[['FC']]
temp[temp < 1] = -1 / temp[temp < 1]
testCountsdf[['FC']] = temp
head(testCountsdf)

,PgrFlox1,PgrFlox2,PgrFlox3,PgrKO1,PgrKO2,PgrKO3,padj,FC
,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>
TrnP,7827,9648,7676,6926,6661,7954,0.7099399,-1.117102
TrnT,46,57,48,30,42,50,0.7913255,-1.188113
CYTB,2818357,3056764,3627036,2714250,3060874,3061130,0.9198277,-1.014453
TrnE,181,272,331,190,241,307,0.9947032,1.003623
ND6,360990,486772,501792,381434,419012,419213,0.8507135,-1.042027
ND5,1264726,1503209,1659052,1104796,1434887,1482613,0.8273932,-1.043338


In [39]:
testCountsThresholdeddf = subset(testCountsdf, padj < 0.05)
testCountsThresholdeddf = testCountsThresholdeddf[order(testCountsThresholdeddf[['FC']], decreasing=TRUE),]

head(testCountsThresholdeddf)
tail(testCountsThresholdeddf)
nrow(testCountsThresholdeddf)

,PgrFlox1,PgrFlox2,PgrFlox3,PgrKO1,PgrKO2,PgrKO3,padj,FC
,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>
Bpifa3,9,30,19,14,15572,6265,9.615476e-05,385.43576
Gtsf1l,15,10,11,11,1682,361,3.503845e-03,56.96074
Zfp990,47,127,150,3049,118,9963,3.093534e-03,43.47686
Gm39385,8,69,29,76,1629,1656,3.464510e-03,32.32008
Gm40378,246,37,356,874,438,10342,3.789558e-02,19.00809
Gm33327,10,0,2,17,73,136,9.305053e-03,18.01639


,PgrFlox1,PgrFlox2,PgrFlox3,PgrKO1,PgrKO2,PgrKO3,padj,FC
,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>
Gm40358,2338,236,2206,97,57,86,3.422606e-04,-18.63492
Slc6a14,27776,365,21270,701,322,516,8.335307e-03,-30.20567
5330429C05Rik,53,43,189,2,4,0,3.927270e-06,-41.87233
4933432I03Rik,719,13,504,9,8,6,1.120908e-03,-51.26640
Drd4,12458,80,8716,136,86,73,1.819209e-03,-68.03490
Ano2,14251,99,4304,53,67,90,6.630990e-04,-90.70434


[1] 1682

In [40]:
# Output results to a CSV
fwrite(testCountsThresholdeddf, "./Pgr_DEGs_P05.csv", row.names=TRUE)

In [41]:
# Make DEseq2 normalized counts table
normalized_counts_table <- counts(dds, normalized = TRUE)
head(normalized_counts_table)

,PgrFlox1,PgrFlox2,PgrFlox3,PgrKO1,PgrKO2,PgrKO3
TrnP,8.601668e+03,9.706262e+03,6.341322e+03,7.762832e+03,6.591654e+03,7.709579e+03
TrnT,5.055279e+01,5.734421e+01,3.965392e+01,3.362474e+01,4.156275e+01,4.846354e+01
CYTB,3.097300e+06,3.075223e+06,2.996379e+06,3.042199e+06,3.029008e+06,2.967064e+06
TrnE,1.989143e+02,2.736425e+02,2.734468e+02,2.129567e+02,2.384910e+02,2.975661e+02
ND6,3.967185e+05,4.897115e+05,4.145420e+05,4.275207e+05,4.146498e+05,4.063309e+05
ND5,1.389901e+06,1.512286e+06,1.370581e+06,1.238283e+06,1.419949e+06,1.437053e+06


In [42]:
# Output results to a CSV
write.csv(normalized_counts_table, "./Pgr_normcounts.csv", row.names=TRUE)

In [43]:
# to make sure we have all the DEseq2 output, let's convert all the results to a dataframe

res_df <- as.data.frame(res)
head(res_df)

,baseMean,log2FoldChange,lfcSE,stat,pvalue,padj
,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>
TrnP,7.785553e+03,-0.159761552,0.19347326,-0.82575522,0.4089430,0.7099399
TrnT,4.520032e+01,-0.248671984,0.39263079,-0.63334815,0.5265063,0.7913255
CYTB,3.034529e+06,-0.020702092,0.07252876,-0.28543285,0.7753125,0.9198277
TrnE,2.491696e+02,0.005216982,0.24731056,0.02109486,0.9831700,0.9947032
ND6,4.249122e+05,-0.059392886,0.12367199,-0.48024526,0.6310530,0.8507135
ND5,1.394676e+06,-0.061206571,0.11372354,-0.53820493,0.5904356,0.8273932


In [44]:
# Output results to a CSV
write.csv(res_df, "./Pgr_DEseq2Results.csv", row.names=TRUE)